# Genome-wide QC + thinning (Google Batch / dsub)

Builds the GRM panel -- one per final `SAMPLE_SET`. Runs *after* `01_ancestry_pca_filter.ipynb` (classification) and `02_build_unified_panel.ipynb` (shared candidate variant list): MAF/HWE are population-specific statistics, so QC needs to run against each `SAMPLE_SET`'s real final membership, but starting from the *same* candidate variant list every `SAMPLE_SET` shares -- not an independently-thinned one per `SAMPLE_SET` like before.

No more raw ACAF access, no Requester Pays workaround -- `02`'s unified panel already lives in this project's own bucket, so plain `--input-recursive`/`--input` work directly. One dsub job per `SAMPLE_SET`: `--keep` + MAF/HWE/geno QC, in a single `plink2` call.

## Prerequisites

`dsub` installed, `gcloud` authenticated, `02_build_unified_panel.ipynb` already run (its merge step, specifically).

In [ ]:
%%bash
set -e

if ! command -v dsub >/dev/null 2>&1; then
  pip install --quiet dsub
fi
dsub --version

echo "--- gcloud config ---"
gcloud config list --format='text(core.project,compute.region)' 2>&1 || true

## Inputs

Same `PROJECT_ID`/`REGION`/`SERVICE_ACCOUNT`/`NETWORK`/`SUBNETWORK`/`WORKSPACE_BUCKET_GS` as everywhere else in this pipeline. `KEEP_PATH` comes from `01_ancestry_pca_filter.ipynb`'s own `final_keep_ids_{SAMPLE_SET}_{prob_tag}.txt`. `UNIFIED_PANEL_DIR_GS` comes from `02_build_unified_panel.ipynb`'s merge step -- also where the staged `plink2` binary lives now (`bin/plink2` inside it), so no separate plink2-staging step is needed in this notebook anymore.

In [ ]:
import os

PROJECT_ID = "wb-swift-sprout-7231"
REGION = "us-central1"
SERVICE_ACCOUNT = "pet-27799165194323faf22e2@wb-swift-sprout-7231.iam.gserviceaccount.com"
NETWORK = f"projects/{PROJECT_ID}/global/networks/network"
SUBNETWORK = f"projects/{PROJECT_ID}/regions/{REGION}/subnetworks/subnetwork"
WORKSPACE_BUCKET_GS = "gs://cloned-shared-env-pilot-wb-swift-sprout-7231"

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
CDR_VERSION = "v9"

# Top-level bucket folder name for this project's outputs -- distinct from
# CDR_VERSION, which keeps its real meaning elsewhere. Fixed literal, matches
# every other notebook in this pipeline.
PROJECT_DIR = "phenotypic_covariance_v9"

# Every final SAMPLE_SET -- must match 01_ancestry_pca_filter.ipynb's own
# SAMPLE_SETS dict (only prob_tag is needed here, to name the keep-list file).
SAMPLE_SETS = {
    "eur_strict": {"prob_tag": "p10"},
    "eur_base":   {"prob_tag": "p50"},
    "eur_loose":  {"prob_tag": "p99"},
    "uniform": {"prob_tag": "uniform"},
    "afr":        {"prob_tag": "p2"},
    "eas":        {"prob_tag": "p90"},
}

SAMPLE_SET = "eur_base"   # <-- change this and rerun for each of the 6 sample sets
_cfg = SAMPLE_SETS[SAMPLE_SET]

ANCESTRY_BUCKET_DIR = f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/01_ancestry_filtering"
ANCESTRY_BUCKET_DIR_GS = f"{WORKSPACE_BUCKET_GS}/{PROJECT_DIR}/01_ancestry_filtering"

KEEP_PATH = f"{ANCESTRY_BUCKET_DIR}/ancestry_pca_filter/final_pca/{SAMPLE_SET}/final_keep_ids_{SAMPLE_SET}_{_cfg['prob_tag']}.txt"
KEEP_PATH_GS = f"{ANCESTRY_BUCKET_DIR_GS}/ancestry_pca_filter/final_pca/{SAMPLE_SET}/final_keep_ids_{SAMPLE_SET}_{_cfg['prob_tag']}.txt"
assert os.path.isfile(KEEP_PATH), f"keep-list not found: {KEEP_PATH!r} -- run 01_ancestry_pca_filter.ipynb first"

# 02_build_unified_panel.ipynb's merged output -- shared starting variant list
# across every SAMPLE_SET, no longer independently re-derived here
UNIFIED_PANEL_DIR_GS = f"{ANCESTRY_BUCKET_DIR_GS}/unified_panel"
UNIFIED_PANEL_NAME = f"unified_panel_{CDR_VERSION}"

# one panel per SAMPLE_SET -- QC/MAF/HWE need to run against each SAMPLE_SET's
# real final membership, not a shared broad proxy
BUCKET_DIR_GS = f"{ANCESTRY_BUCKET_DIR_GS}/genome_wide_panel_{SAMPLE_SET}"

# per-task machine -- reused from the old per-chromosome pipeline's sizing as a
# starting point (real chr1-4 OOM/disk history there). Retune if this single,
# much-smaller-input job doesn't actually need this much.
MACHINE_VCPUS = 16
MEMORY_MB = 55000

print(UNIFIED_PANEL_DIR_GS)
print(BUCKET_DIR_GS)

## Submit (dsub)

One job per `SAMPLE_SET`: `--keep` + MAF/HWE/geno QC in a single `plink2` call against the unified panel, plus a `--make-bed` export (PLINK 1.9's `--parallel` split, used by the GRM step, needs bed/bim/fam). No more 22-chromosome staging -- the unified panel is already merged genome-wide.

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION" "$SERVICE_ACCOUNT" "$NETWORK" "$SUBNETWORK" "$UNIFIED_PANEL_DIR_GS" "$UNIFIED_PANEL_NAME" "$KEEP_PATH_GS" "$BUCKET_DIR_GS" "$SAMPLE_SET" "$CDR_VERSION" "$MACHINE_VCPUS" "$MEMORY_MB" "$ANCESTRY_BUCKET_DIR_GS"
set -e
PROJECT_ID=$1
REGION=$2
SERVICE_ACCOUNT=$3
NETWORK=$4
SUBNETWORK=$5
UNIFIED_PANEL_DIR_GS=$6
UNIFIED_PANEL_NAME=$7
KEEP_PATH_GS=$8
BUCKET_DIR_GS=$9
SAMPLE_SET=${10}
CDR_VERSION=${11}
MACHINE_VCPUS=${12}
MEMORY_MB=${13}
ANCESTRY_BUCKET_DIR_GS=${14}

LOGGING_GS="${ANCESTRY_BUCKET_DIR_GS}/dsub_logs"
OUT_NAME="genome_wide_thinned_${CDR_VERSION}_${SAMPLE_SET}"

dsub \
  --provider google-batch \
  --project "$PROJECT_ID" \
  --regions "$REGION" \
  --logging "$LOGGING_GS" \
  --service-account "$SERVICE_ACCOUNT" \
  --network "$NETWORK" \
  --subnetwork "$SUBNETWORK" \
  --use-private-address \
  --image "gcr.io/google.com/cloudsdktool/cloud-sdk:slim" \
  --name "genome-wide-qc-${SAMPLE_SET}" \
  --machine-type "n1-standard-${MACHINE_VCPUS}" \
  --disk-size 300 \
  --input-recursive UNIFIED_PANEL="$UNIFIED_PANEL_DIR_GS" \
  --input KEEP_PATH="$KEEP_PATH_GS" \
  --env UNIFIED_PANEL_NAME="$UNIFIED_PANEL_NAME" \
  --env OUT_NAME="$OUT_NAME" \
  --env MACHINE_VCPUS="$MACHINE_VCPUS" \
  --env MEMORY_MB="$MEMORY_MB" \
  --output-recursive OUT_DIR="$BUCKET_DIR_GS" \
  --command '
    set -e
    PLINK_BIN="${UNIFIED_PANEL}/bin/plink2"
    chmod +x "$PLINK_BIN"

    "$PLINK_BIN" \\
      --pfile "${UNIFIED_PANEL}/${UNIFIED_PANEL_NAME}" \\
      --keep "$KEEP_PATH" \\
      --maf 0.01 \\
      --hwe 1e-6 0.001 keep-fewhet \\
      --geno 0.05 \\
      --nonfounders \\
      --threads "$MACHINE_VCPUS" \\
      --memory "$MEMORY_MB" \\
      --make-pgen \\
      --out "${OUT_DIR}/${OUT_NAME}"

    echo "SAMPLE_SET variant count:"
    grep -vc "^##" "${OUT_DIR}/${OUT_NAME}.pvar"
    echo "SAMPLE_SET sample count:"
    wc -l < "${OUT_DIR}/${OUT_NAME}.psam"

    # PLINK 1.9 (the GRM step) does not read pgen -- export a bed/bim/fam copy too
    "$PLINK_BIN" \\
      --pfile "${OUT_DIR}/${OUT_NAME}" \\
      --make-bed \\
      --threads "$MACHINE_VCPUS" \\
      --out "${OUT_DIR}/${OUT_NAME}_bed"
  ' \
  > /tmp/qc_job_id.txt

cat /tmp/qc_job_id.txt

## Check status

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION"
set -e
PROJECT_ID=$1
REGION=$2
JOB_ID=$(cat /tmp/qc_job_id.txt)

dstat --provider google-batch --project "$PROJECT_ID" --location "$REGION" --jobs "$JOB_ID" --users 'jupyter' --status '*' --full

## Next steps

`04_final_pca.ipynb` reads this `SAMPLE_SET`'s panel directly (`genome_wide_thinned_{CDR_VERSION}_{SAMPLE_SET}` -- already restricted/QC'd, no further `--keep` needed).